| 指标\水印名称 | LSB  | DWT  | DWT_SVD | MBRS | CIN  | TreeRing | GaussianShading | GaussianMarker |
|---------|------|------|---------|------|------|----------|-----------------|----------------|
| ACC     | 99.7 | 99.6 | 99.8    | 100  | 100  |          |                 |                |
| FPR     | 0.6  | 0    | 0       | 0    | 0    |          |                 |                |
| TP      | 1000 | 992  | 996     | 1000 | 1000 |          |                 |                |
| TN      | 0    | 8    | 4       | 0    | 0    |          |                 |                |
| FP      | 6    | 0    | 0       | 0    | 0    |          |                 |                |
| FN      | 994  | 1000 | 1000    | 1000 | 1000 |          |                 |                |

## 水印嵌入

### 实验环境与水印导入

In [1]:
# ============== 导入核心库 ================
from watermarks.core import WatermarkerFactory
import watermarks.implementations
import importlib
import pkgutil
from utils.monitor import WatermarkProfiler  # type: ignore

# ============== 动态加载所有水印实现模块 ================
imported_modules = {}
package = watermarks.implementations
prefix = package.__name__ + "."

for _, name, is_pkg in pkgutil.iter_modules(package.__path__, prefix):
    module = importlib.import_module(name)
    short_name = name.split('.')[-1]
    imported_modules[short_name] = module
    print(f"已导入模块: {short_name}")

print("所有水印实现模块加载完毕。")
print(f"可用模块列表: {list(imported_modules.keys())}")

# ============== 水印实验执行函数 ================
# 进行硬编码：将seed、sampling_retio、prompt_folder等参数直接写入函数中，确保每次实验的一致性
def run_watermark_experiment(method_name, method_params, input_folder, output_folder, threshold=0.05, mix_ratio=0.5, prompt_folder=None, mode='both'):
    """
    运行指定水印方法的嵌入和/或提取实验
    
    参数:
        method_name: 水印方法名称 (如 'lsb', 'tree_ring' 等)
        method_params: 该水印方法所需的参数字典
        mode: 运行模式 ('both': 嵌入+提取, 'embed_only': 仅嵌入, 'extract_only': 仅提取)
    """
    print(f"\n{'='*50}")
    print(f"开始测试水印方法: {method_name}")
    print(f"参数配置: {method_params}")
    print(f"运行模式: {mode}")
    print(f"{'='*50}")
    
    # 设置默认参数
    method_params['seed'] = 42
    method_params['sampling_ratio'] = 0.05
    method_params['prompt_folder'] = prompt_folder
    
    # 创建水印器实例
    watermarker = WatermarkerFactory.create(
        name=method_name,
        params=method_params
    )
    
    result = None
    
    # 模式判断和执行
    if mode in ['both', 'embed_only']:
        # 执行水印嵌入
        print("正在执行水印嵌入...")
        with WatermarkProfiler(f"{method_name}_embedding"):
            watermarker.embed_batch(
                input_dir=input_folder,
                output_dir=output_folder
            )
        print("水印嵌入完成。")
    
    if mode in ['both', 'extract_only']:
        # 执行水印提取与检测
        print("正在执行水印提取与检测...")
        with WatermarkProfiler(f"{method_name}_extraction"):
            result = watermarker.extract_batch(
                image_dir_to_check=output_folder if mode == 'both' else input_folder,
                clean_dir=input_folder if mode == 'extract_only' else input_folder,
                threshold=threshold,
                mix_ratio=mix_ratio,
            )
        
        print(f"水印检测结果: {result}")
    
    print(f"{'='*50}")
    print(f"方法 {method_name} 测试完成。")
    print(f"{'='*50}\n")
    
    return result

已导入模块: cin
已导入模块: dwt
已导入模块: dwt_svd
已导入模块: gm
已导入模块: gs
已导入模块: lsb
已导入模块: mbrs
已导入模块: tree_ring
所有水印实现模块加载完毕。
可用模块列表: ['cin', 'dwt', 'dwt_svd', 'gm', 'gs', 'lsb', 'mbrs', 'tree_ring']


### LSB嵌入

实验配置：

In [5]:
# 路径参数
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/lsb_watermarked"

SECRET_32 = [1,0,1,0, 0,1,0,1, 1,1,0,0, 0,0,1,1,
             1,0,0,1, 0,1,1,0, 1,1,1,0, 0,0,0,1]

In [6]:
print("执行LSB水印实验...")
lsb_params = {
    "secret": SECRET_32
}
lsb_result = run_watermark_experiment(
    method_name="lsb",
    method_params=lsb_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)
print("LSB水印实验完成。")

执行LSB水印实验...

开始测试水印方法: lsb
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1]}
运行模式: both
正在执行水印嵌入...
--- [Profiler] Start monitoring: lsb_embedding ---
[Info] Sampling: 1000/20000 images (Ratio: 0.05)
[LSB] Embedding start. Processing 1000 files...
--- [Profiler] Result for lsb_embedding ---
    Max RAM Usage : 664.03 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印嵌入完成。
正在执行水印提取与检测...
--- [Profiler] Start monitoring: lsb_extraction ---
[LSB] Extraction start. Checking 2000 files (Mix Ratio: 0.5)
[LSB] Done. Acc: 99.70% | FPR: 0.60%
Stats: TP=1000, FN=0, FP=6, TN=994
--- [Profiler] Result for lsb_extraction ---
    Max RAM Usage : 663.60 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印检测结果: {'total': 2000, 'metrics': {'TP': 1000, 'FP': 6, 'TN': 994, 'FN': 0}, 'accuracy': 0.997, 'fpr': 0.006, 'details': {'CLN_013074.png': {'distance': 0.42857142857142855, 'is_watermarke

### dwt嵌入

实验配置：

In [7]:
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/dwt_watermarked"

In [8]:
print("执行DWT水印实验...")
dwt_params = {
    "secret": SECRET_32
}
dwt_result = run_watermark_experiment(
    method_name="dwt",
    method_params=dwt_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)
print("DWT水印实验完成。")

执行DWT水印实验...

开始测试水印方法: dwt
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1]}
运行模式: both
正在执行水印嵌入...
--- [Profiler] Start monitoring: dwt_embedding ---
[Info] Sampling: 1000/20000 images (Ratio: 0.05)
[DWTQIMWatermarker] Embedding start. Processing 1000 files...
--- [Profiler] Result for dwt_embedding ---
    Max RAM Usage : 674.30 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印嵌入完成。
正在执行水印提取与检测...
--- [Profiler] Start monitoring: dwt_extraction ---
[DWTQIMWatermarker] Extraction start. Checking 2000 files (Mix Ratio: 0.5)
[DWTQIMWatermarker] Done. Acc: 99.60% | FPR: 0.00%
Stats: TP=992, FN=8, FP=0, TN=1000
--- [Profiler] Result for dwt_extraction ---
    Max RAM Usage : 674.98 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印检测结果: {'total': 2000, 'metrics': {'TP': 992, 'FP': 0, 'TN': 1000, 'FN': 8}, 'accuracy': 0.996, 'fpr': 0.0, 'details': {'CLN_013074.png': {'distan

### dwt_svd嵌入

实验配置：

In [9]:
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/dwt_svd_watermarked"

In [10]:
print("执行DWT-SVD水印实验...")
dwt_svd_params = {
    "secret": SECRET_32
}
dwt_svd_result = run_watermark_experiment(
    method_name="dwt_svd",
    method_params=dwt_svd_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)
print("DWT-SVD水印实验完成。")

执行DWT-SVD水印实验...

开始测试水印方法: dwt_svd
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1]}
运行模式: both
正在执行水印嵌入...
--- [Profiler] Start monitoring: dwt_svd_embedding ---
[Info] Sampling: 1000/20000 images (Ratio: 0.05)
[DWTSVDWatermarker] Embedding start. Processing 1000 files...
--- [Profiler] Result for dwt_svd_embedding ---
    Max RAM Usage : 674.68 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印嵌入完成。
正在执行水印提取与检测...
--- [Profiler] Start monitoring: dwt_svd_extraction ---
[DWTSVDWatermarker] Extraction start. Checking 2000 files (Mix Ratio: 0.5)
[DWTSVDWatermarker] Done. Acc: 99.80% | FPR: 0.00%
Stats: TP=996, FN=4, FP=0, TN=1000
--- [Profiler] Result for dwt_svd_extraction ---
    Max RAM Usage : 673.30 MB
    Max VRAM Usage: 0.00 MB
-----------------------------------------
水印检测结果: {'total': 2000, 'metrics': {'TP': 996, 'FP': 0, 'TN': 1000, 'FN': 4}, 'accuracy': 0.998, 'fpr': 0.0, 'details': {'C

### MBRS嵌入

实验配置：

In [2]:
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/mbrs_watermarked"
SECRET_64 = [1,0,1,0, 0,1,0,1, 1,1,0,0, 0,0,1,1,
             1,0,0,1, 0,1,1,0, 1,1,1,0, 0,0,0,1,
             0,1,0,0, 1,0,1,1, 1,0,0,0, 0,1,1,1,
             1,1,0,1, 0,0,1,0, 1,0,1,0, 0,1,0,1] 

In [3]:
print("执行MBRS水印实验...")
mbrs_params = {
    "secret": SECRET_64
}
mbrs_result = run_watermark_experiment(
    method_name="mbrs",
    method_params=mbrs_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)

执行MBRS水印实验...

开始测试水印方法: mbrs
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1]}
运行模式: both
正在执行水印嵌入...
--- [Profiler] Start monitoring: mbrs_embedding ---
[Proxy] Calling mbrs [embed]...
[Worker] Imported mbrs
[MBRS] Loading Network on cuda...
0.0001
[MBRS] Weights loaded from D:\graduation\computer\Watermark\models\watermarker\MBRS\results\MBRS_256_m256\models\EC_42.pth
[Info] Sampling: 1000/20000 images (Ratio: 0.05)
[MBRSWatermarker] Embedding start. Processing 1000 files...

--- [Profiler] Result for mbrs_embedding ---
    Max RAM Usage : 1823.07 MB
    Max VRAM Usage: 308.20 MB
-----------------------------------------
水印嵌入完成。
正在执行水印提取与检测...
--- [Profiler] Start monitoring: mbrs_extraction ---
[Proxy] Calling mbrs [extract]...
[Worker] Imported mbrs
[MBRS] Loading Network on cuda...
0.0001
[MBRS] Weights loaded from D:\gra

### CIN嵌入

In [13]:
input_folder = "D:/graduation/computer/Watermark/dataset/origin"
output_folder = "D:/graduation/computer/Watermark/results/cin_watermarked"
SECRET_30 = [1,0,1,0, 0,1,0,1, 1,1,0,0, 0,0,1,1,
             1,0,0,1, 0,1,1,0, 1,1,1,0, 0,0]
    

In [14]:
cin_params = {
    "secret": SECRET_30
}
print("执行CIN水印实验...")
cin_result = run_watermark_experiment(
    method_name="cin",
    method_params=cin_params,
    input_folder=input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5
)
print("CIN水印实验完成。")

执行CIN水印实验...

开始测试水印方法: cin
参数配置: {'secret': [1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0]}
运行模式: both
正在执行水印嵌入...
--- [Profiler] Start monitoring: cin_embedding ---
[Proxy] Calling cin [embed]...
[Worker] Imported cin
[CIN] Initializing Network on cuda...
[CIN] Loading weights from D:/graduation/computer/Watermark/models/watermarker/CIN/pth/cinNet&nsmNet.pth
[Info] Sampling: 1000/20000 images (Ratio: 0.05)
[CINWatermarker] Embedding start. Processing 1000 files...

--- [Profiler] Result for cin_embedding ---
    Max RAM Usage : 2013.09 MB
    Max VRAM Usage: 420.20 MB
-----------------------------------------
水印嵌入完成。
正在执行水印提取与检测...
--- [Profiler] Start monitoring: cin_extraction ---
[Proxy] Calling cin [extract]...



==================== [Worker 报错详情开始] ====================
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)

==================== [Worker 报错详情结束] ====================



[Worker] Imported cin
[CIN] Initializing Network on cuda...
[CIN] Loading weights from D:/graduation/computer/Watermark/models/watermarker/CIN/pth/cinNet&nsmNet.pth
[CINWatermarker] Extraction start. Checking 2000 files (Mix Ratio: 0.5)
[CINWatermarker] Done. Acc: 100.00% | FPR: 0.00%
Stats: TP=1000, FN=0, FP=0, TN=1000

[Proxy] Failed to parse JSON from worker output. Returning empty dict.
[Proxy] Calling cin [extract]...



==================== [Worker 报错详情开始] ====================
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)

==================== [Worker 报错详情结束] ====================



[Worker] Imported cin
[CIN] Initializing Network on cuda...
[CIN] Loading weights from D:/graduation/computer/Watermark/models/watermarker/CIN/pth/cinNet&nsmNet.pth
[CINWatermarker] Extraction start. Checking 2000 files (Mix Ratio: 0.5)
[CINWatermarker] Done. Acc: 100.00% | FPR: 0.00%
Stats: TP=1000, FN=0, FP=0, TN=1000

[Proxy] Failed to parse JSON from worker output. Returning empty dict.
--- [Profiler] Result for cin_extraction ---
    Max RAM Usage : 2022.43 MB
    Max VRAM Usage: 424.07 MB
-----------------------------------------
水印检测结果: {}
方法 cin 测试完成。

CIN水印实验完成。



==================== [Worker 报错详情开始] ====================
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\anaconda3\envs\cin\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)

==================== [Worker 报错详情结束] ====================



### Treering嵌入

实验配置：

In [2]:
input_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/origin"
output_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/tree_ring_watermarked"
prompt_input_folder = 'C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts'

In [3]:
print("执行Tree Ring水印实验...")
tree_ring_params = {
            'image_length': 512,
            'max_images':1000
}
tree_ring_result = run_watermark_experiment(
    method_name="tree_ring",
    method_params=tree_ring_params,
    input_folder=prompt_input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5,
    mode='embed_only'
)

执行Tree Ring水印实验...

开始测试水印方法: tree_ring
参数配置: {'image_length': 512, 'max_images': 1000}
运行模式: embed_only
正在执行水印嵌入...
--- [Profiler] Start monitoring: tree_ring_embedding ---
[Proxy] Calling tree_ring [embed]...
[Proxy] Command: C:\ProgramData\anaconda3\envs\tree_ring\python.exe -u c:\Users\Administrator\Desktop\file\graduation\computer\Watermark\watermarks\worker.py --name tree_ring --mode embed --input C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts --params {"image_length": 512, "max_images": 1000, "prompt_folder": null, "sampling_ratio": 0.05, "seed": 42} --output C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/tree_ring_watermarked
--- [Profiler] Result for tree_ring_embedding ---
    Max RAM Usage : 6298.36 MB
    Max VRAM Usage: 5108.67 MB
-----------------------------------------
水印嵌入完成。
方法 tree_ring 测试完成。



### Gaussian_Shading嵌入

实验配置：

In [2]:
input_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/origin"
output_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/gs_watermarked"
prompt_input_folder = 'C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts'

In [3]:
print("执行GS水印实验...")
gs_params = {
    'image_length': 512,
    'max_images':1000
}
gs_result = run_watermark_experiment(
    method_name="gs",
    method_params=gs_params,
    input_folder=prompt_input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5,
    mode='embed_only'
)

执行GS水印实验...

开始测试水印方法: gs
参数配置: {'image_length': 512, 'max_images': 1000}
运行模式: embed_only
正在执行水印嵌入...
--- [Profiler] Start monitoring: gs_embedding ---
[Proxy] Calling gs [embed]...
[Proxy] Command: C:\ProgramData\anaconda3\envs\gs\python.exe -u c:\Users\Administrator\Desktop\file\graduation\computer\Watermark\watermarks\worker.py --name gs --mode embed --input C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts --params {"image_length": 512, "max_images": 1000, "prompt_folder": null, "sampling_ratio": 0.05, "seed": 42} --output C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/gs_watermarked
--- [Profiler] Result for gs_embedding ---
    Max RAM Usage : 5694.26 MB
    Max VRAM Usage: 3828.91 MB
-----------------------------------------
水印嵌入完成。
方法 gs 测试完成。



### GaussMarker嵌入

实验配置：

In [5]:
input_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/origin"
output_folder = "C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/gm_watermarked"
prompt_input_folder = 'C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts'
gm_params = {
    'image_length': 512,
    'max_images':1000,
    'seed': 42,
    'train_gnr_after_embed':True,
    'gnr_batch_size':16
}

In [6]:
gm_result = run_watermark_experiment(
    method_name="gm",
    method_params=gm_params,
    input_folder=prompt_input_folder,
    output_folder=output_folder,
    threshold=0.05,
    mix_ratio=0.5,
    mode='embed_only'
)


开始测试水印方法: gm
参数配置: {'image_length': 512, 'max_images': 1000, 'seed': 42, 'train_gnr_after_embed': True, 'gnr_batch_size': 16}
运行模式: embed_only
正在执行水印嵌入...
--- [Profiler] Start monitoring: gm_embedding ---
[Proxy] Calling gm [embed]...
[Proxy] Command: C:\ProgramData\anaconda3\envs\gm\python.exe -u c:\Users\Administrator\Desktop\file\graduation\computer\Watermark\watermarks\worker.py --name gm --mode embed --input C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/dataset/prompts/stable_diffusion_prompts --params {"image_length": 512, "max_images": 1000, "train_gnr_after_embed": true, "gnr_batch_size": 16, "prompt_folder": null, "sampling_ratio": 0.05, "seed": 42} --output C:/Users/Administrator/Desktop/file/graduation/computer/Watermark/results/gm_watermarked
--- [Profiler] Result for gm_embedding ---
    Max RAM Usage : 6484.10 MB
    Max VRAM Usage: 3770.68 MB
-----------------------------------------
水印嵌入完成。
方法 gm 测试完成。

